In [ ]:
import pandas as pd
import numpy as np

In [ ]:
END_PATH  = "../../data/us_leagues/csv/end_of_season/actual/end_of_season_us.csv"   # league,season,team_id,...,rank
GAMES_PATH = "../../data/us_leagues/csv/game_by_game/actual/us_combined_data.csv"  # season,date,league,team1,team2,result,score1,score2
end = pd.read_csv(END_PATH)
games = pd.read_csv(GAMES_PATH)

In [ ]:
rng_seed = 42     # set None for non-deterministic ties
tie_p    = 0.5    # probability team1 wins when ranks are equal

In [ ]:
# --- merge ranks from same league & season; every team must exist in EOS ---
sim = games.merge(
    end[["league","season","team_id","rank"]]
        .rename(columns={"team_id":"team1","rank":"team1_rank"}),
    on=["league","season","team1"], how="inner"
).merge(
    end[["league","season","team_id","rank"]]
        .rename(columns={"team_id":"team2","rank":"team2_rank"}),
    on=["league","season","team2"], how="inner"
)

In [ ]:
assert len(sim) == len(games), "Some games didn't match EOS ranks — check league/season/team ids."

In [ ]:
# team1 better -> +1; team2 better -> -1; equal -> RNG coin (P(+1)=tie_p)
sim_res = np.where(sim["team1_rank"] < sim["team2_rank"],  1,
           np.where(sim["team1_rank"] > sim["team2_rank"], -1, 0))

ties = (sim_res == 0)
rng  = np.random.default_rng(rng_seed) if rng_seed is not None else np.random.default_rng()
sim_res[ties] = np.where(rng.random(ties.sum()) < tie_p, 1, -1)

In [ ]:
# --- finalize output DataFrame (games unchanged) ---
sim["sim_result"] = sim_res.astype(int)
sim["rank_tie"]   = ties

sim = sim[[
    "league","season","date","team1","team2",
    "team1_rank","team2_rank","sim_result","rank_tie",
    "result","score1","score2"
]]

In [ ]:
sim

In [ ]:
result_df = games.copy()
result_df['result'] = sim['sim_result']

In [ ]:
result_df

In [ ]:
result_df.to_csv("../../data/us_leagues/csv/game_by_game/pure_skill/us_combined_pure_skill.csv", index=False)